In [1]:
from pydantic import BaseModel, Field, model_validator, ValidationInfo
from typing import List, Optional, Literal, Type
from functools import partial
from enum import Enum
import instructor
from openai import AzureOpenAI
import pandas as pd
import os
from openpyxl import Workbook
from tqdm import tqdm
import difflib
from entity_test import PSOM_total,Neuro_Deficit_Score_Severity,Neuro_Deficit_Type,follow_up_status,post_discharge_rehabilitation, QuestionAnswer


# Define your Azure OpenAI configuration
model = "gpt4o_cursor"
engine = "gpt4o_cursor"
endpoint = "https://gpt4o-xj.openai.azure.com/"

api_version = "2024-02-15-preview"
api_type = "azure"

# key_name = "AZURE_KEY"
api_key = '15f19bd128a54ec7b2a10f46299272ec'

# Create the Azure OpenAI client
client = AzureOpenAI(
    api_version=api_version,
    api_key=api_key,
    azure_endpoint=endpoint,
    azure_deployment=engine
)

# Patch the client with instructor
client = instructor.patch(client)

# Optionally, create a partial function with default parameters
in_client = partial(
    client.chat.completions.create,
    model=model,
    temperature=0,
    max_retries=2  # Retries in case of API issues
)

# Directory Configuration
output_directory = "output/"
note_directory = "notes/"
results_file_name = 'results.xlsx'

In [2]:
SYSTEM_PROMPT = """
You are a clinical note reviewer. Given a clinical note and a question, follow these steps:

1. Carefully read the clinical note.
2. Identify any exact quotes (phrases) from the note that support a potential answer.
3. From these quotes, infer a fact that answers the question.
4. Based on the fact, answer the question.

Important:
- Only use direct quotes from the clinical note.
- If there is no evidence in the note, answer Unknown.

Return your output as a JSON object with:
- answer
- fact
- substring_quote (list of strings, must come directly from the note)
"""

# 5. Explain your reasoning.

In [7]:
def ask_structured_answer(
    notes: str,
    schema_class: Type[BaseModel]
) -> BaseModel:
    # prompt = schema_class.__doc__
    cleaned_notes = notes.strip()

    response = in_client(
        response_model=schema_class,
        messages=[
            {"role": "system", "content": (
                "You are a clinical information extractor. "
                "Your goal is to fill the fields in the given schema by reading the note and reasoning based on direct quotes. "
                "You must only rely on exact quotes from the note."
            )},
            {"role": "user", "content": f"CLINICAL NOTE:\n{cleaned_notes}"},
            # {"role": "user", "content": f"TASK:\n{prompt.strip()}"},
        ],
        validation_context={"text_chunk": notes},
    )
    return response

with open(note_directory + "30128-1.txt", "r") as f:
    notes = f.read()

response = ask_structured_answer(notes, post_discharge_rehabilitation)


In [8]:
response

post_discharge_rehabilitation(post_discharge_rehabilitation=<YesNo.YES: 'Yes'>, post_discharge_rehabilitation_type=['PT', 'OT', 'ST'], answer_facts=[Fact(fact='After discharge, she went to IPR where she had significant improvement in her walking and talking, but is still not back to normal.', substring_quote=['After discharge, she went to IPR where she had significant improvement in her walking and talking, but is still not back to normal.']), Fact(fact='Still waiting for authorization to continue PT, OT, and ST.', substring_quote=['Still waiting for authorization to continue PT, OT, and ST.']), Fact(fact='Pending extension of PT/OT/ST', substring_quote=['Pending extension of PT/OT/ST']), Fact(fact='She finished her therapy services three weeks ago and they are applying for an extension of benefits on her behalf.', substring_quote=['She finished her therapy services three weeks ago and they are applying for an extension of benefits on her behalf.'])])